### Dependencies

In [ ]:
import random

from typing import Literal, TypedDict

from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np


### Tool Payload Types

In [ ]:
class CartItem(BaseModel):
    """A single line in the shopping cart the warehouse tools operate on.

    A model rather than a TypedDict: this is the tool's input, so the LLM fills
    it in. The field descriptions become part of the JSON schema the model reads,
    and the values are validated at construction instead of trusted.
    """

    product_id: str = Field(description="The parent ASIN of the product to check.")
    quantity: int = Field(gt=0, description="How many units the customer wants.")


class WarehouseSummary(TypedDict):
    """Identifying fields of a warehouse, without any availability numbers."""

    warehouse_id: str
    warehouse_name: str
    warehouse_location: str


class ItemAvailability(TypedDict):
    """How much of one requested item a single warehouse can supply."""

    product_id: str
    requested: int
    available: int
    can_fulfill_completely: bool
    can_fulfill_partially: bool


class WarehouseAvailability(WarehouseSummary):
    """Per-warehouse breakdown: every requested item plus the warehouse verdict."""

    items: list[ItemAvailability]
    can_fulfill_all: bool
    has_partial: bool


class UnavailableItem(TypedDict):
    """An item whose requested quantity exceeds total stock across all warehouses."""

    product_id: str
    requested: int
    total_available_across_warehouses: int
    shortage: int


class AvailabilityCheck(TypedDict):
    """Result of checking a cart against every warehouse."""

    can_fulfill_completely: bool
    warehouses_full_fulfillment: list[WarehouseSummary]
    warehouses_partial_fulfillment: list[WarehouseSummary]
    unavailable_items: list[UnavailableItem]
    details: list[WarehouseAvailability]


class ReservationRequest(BaseModel):
    """One warehouse-specific reservation the LLM asks the tool to make.

    Unlike CartItem this names a warehouse, so the caller has already decided
    where each item comes from - typically from a check_warehouse_availability
    result.
    """

    warehouse_id: str = Field(description="The warehouse to reserve from.")
    product_id: str = Field(description="The parent ASIN of the product to reserve.")
    quantity: int = Field(gt=0, description="How many units to reserve.")


class ReservedItem(TypedDict):
    """An item successfully reserved at one warehouse."""

    product_id: str
    quantity: int
    warehouse_id: str
    warehouse_name: str
    warehouse_location: str


class FailedReservation(TypedDict):
    """An item that could not be reserved, and why."""

    product_id: str
    warehouse_id: str
    requested: int
    available: int
    reason: Literal["insufficient_stock", "not_in_warehouse"]


class ReservationSucceeded(TypedDict):
    """Every requested item was reserved and the transaction committed."""

    status: Literal["reserved"]
    reserved_items: list[ReservedItem]


class ReservationFailed(TypedDict):
    """At least one item could not be reserved, so the whole batch rolled back.

    There is deliberately no reserved_items key: the reservation is all-or-nothing,
    so on this branch nothing is held, however far the transaction got before the
    failing item.
    """

    status: Literal["rejected"]
    failed_items: list[FailedReservation]


# Discriminated on the literal "status" - narrow with an == comparison, since a
# truthiness check on a TypedDict subscript does not narrow the union.
ReservationResult = ReservationSucceeded | ReservationFailed


### Item Availability Check Tool

In [ ]:
shopping_cart = [
    CartItem(product_id="B00005J5B4", quantity=1),
    CartItem(product_id="B00005J5B4", quantity=3),
]


In [ ]:
def check_warehouse_availability(items: list[CartItem]) -> AvailabilityCheck:
    """Check availability of items across warehouses, including partial fulfillment options.

    Args:
        items: A list of items to check. Each item is a CartItem with a product_id
            and a quantity.

    Returns:
        A dictionary containing:
        - can_fulfill_completely: bool indicating if all items can be fulfilled from at least one warehouse
        - warehouses_full_fulfillment: list of warehouses that can fulfill the entire order
        - warehouses_partial_fulfillment: list of warehouses with partial availability
        - unavailable_items: list of items that cannot be fulfilled from any warehouse
        - details: detailed breakdown per warehouse with availability for each item
    """

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password",
    )

    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            result: AvailabilityCheck = {
                "can_fulfill_completely": False,
                "warehouses_full_fulfillment": [],
                "warehouses_partial_fulfillment": [],
                "unavailable_items": [],
                "details": [],
            }

            # Check each warehouse for availability
            warehouse_query = """
                SELECT DISTINCT warehouse_id, warehouse_name, warehouse_location
                FROM warehouses.inventory
            """
            cursor.execute(warehouse_query)
            warehouses = cursor.fetchall()

            for warehouse in warehouses:
                warehouse_can_fulfill_all = True
                has_any_availability = False
                warehouse_details: WarehouseAvailability = {
                    "warehouse_id": warehouse["warehouse_id"],
                    "warehouse_name": warehouse["warehouse_name"],
                    "warehouse_location": warehouse["warehouse_location"],
                    "items": [],
                    "can_fulfill_all": False,
                    "has_partial": False,
                }

                for item in items:
                    product_id = item.product_id
                    requested_quantity = item.quantity

                    # Check availability in this warehouse
                    availability_query = """
                        SELECT product_id, total_quantity, reserved_quantity, available_quantity
                        FROM warehouses.inventory
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(
                        availability_query, (warehouse["warehouse_id"], product_id)
                    )
                    inventory = cursor.fetchone()

                    # int() at the boundary: psycopg2 hands back Any, so this is the
                    # only place the declared int types are actually enforced.
                    available_qty = (
                        int(inventory["available_quantity"]) if inventory else 0
                    )

                    item_detail: ItemAvailability = {
                        "product_id": product_id,
                        "requested": requested_quantity,
                        "available": available_qty,
                        "can_fulfill_completely": available_qty >= requested_quantity,
                        "can_fulfill_partially": available_qty > 0
                        and available_qty < requested_quantity,
                    }

                    warehouse_details["items"].append(item_detail)

                    # Track if warehouse can fulfill this item completely
                    if available_qty < requested_quantity:
                        warehouse_can_fulfill_all = False

                    # Track if warehouse has any availability for any item
                    if available_qty > 0:
                        has_any_availability = True

                # Categorize warehouse
                if warehouse_can_fulfill_all:
                    warehouse_details["can_fulfill_all"] = True
                    result["warehouses_full_fulfillment"].append(
                        {
                            "warehouse_id": warehouse["warehouse_id"],
                            "warehouse_name": warehouse["warehouse_name"],
                            "warehouse_location": warehouse["warehouse_location"],
                        }
                    )
                elif has_any_availability:
                    warehouse_details["has_partial"] = True
                    result["warehouses_partial_fulfillment"].append(
                        {
                            "warehouse_id": warehouse["warehouse_id"],
                            "warehouse_name": warehouse["warehouse_name"],
                            "warehouse_location": warehouse["warehouse_location"],
                        }
                    )

                result["details"].append(warehouse_details)

            # Check if any items cannot be fulfilled from any warehouse
            for item in items:
                product_id = item.product_id
                requested_quantity = item.quantity

                # Get total available quantity across all warehouses
                total_available_query = """
                    SELECT product_id, SUM(available_quantity) as total_available
                    FROM warehouses.inventory
                    WHERE product_id = %s
                    GROUP BY product_id
                """
                cursor.execute(total_available_query, (product_id,))
                total_available = cursor.fetchone()

                # SUM() comes back as Decimal for a numeric column, so coerce here too.
                total_available_qty = (
                    int(total_available["total_available"]) if total_available else 0
                )

                if total_available_qty < requested_quantity:
                    result["unavailable_items"].append(
                        {
                            "product_id": product_id,
                            "requested": requested_quantity,
                            "total_available_across_warehouses": total_available_qty,
                            "shortage": requested_quantity - total_available_qty,
                        }
                    )

            result["can_fulfill_completely"] = (
                len(result["warehouses_full_fulfillment"]) > 0
                and len(result["unavailable_items"]) == 0
            )

            return result

    finally:
        conn.close()


In [ ]:
shopping_cart = [
    CartItem(product_id="B0BWJMKC31", quantity=1),
    CartItem(product_id="B088MG3SSB", quantity=3),
    CartItem(product_id="B0B7XH7VSW", quantity=9),
]



In [ ]:
check_warehouse_availability(shopping_cart)

In [ ]:
shopping_cart = [
    CartItem(product_id="B0BWJMKC31", quantity=300),
    CartItem(product_id="B088MG3SSB", quantity=3),
    CartItem(product_id="B0B7XH7VSW", quantity=9),
]


In [ ]:
check_warehouse_availability(shopping_cart)

### Item Reservation Tool

In [ ]:
def reserve_warehouse_items(
    reservations: list[ReservationRequest],
) -> ReservationResult:
    """Reserve items from multiple warehouses in a single transaction.

    Args:
        reservations: A list of reservations. Each reservation is a ReservationRequest
            naming the warehouse to reserve from, the product, and the quantity.

    Returns:
        Either a ReservationSucceeded with status "reserved" and the committed
        reserved_items, or a ReservationFailed with status "rejected" and the
        failed_items that caused the rollback. The reservation is all-or-nothing,
        so the rejected branch carries no reserved items at all.
    """

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="tools_user",
        password="tools_user_password",
    )
    conn.autocommit = False  # Use transaction

    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            reserved_items: list[ReservedItem] = []
            failed_items: list[FailedReservation] = []

            for reservation in reservations:
                warehouse_id = reservation.warehouse_id
                product_id = reservation.product_id
                quantity = reservation.quantity

                # Check and lock the inventory row
                check_query = """
                    SELECT warehouse_id, product_id, warehouse_name, warehouse_location, 
                           total_quantity, reserved_quantity, available_quantity
                    FROM warehouses.inventory
                    WHERE warehouse_id = %s AND product_id = %s
                    FOR UPDATE
                """
                cursor.execute(check_query, (warehouse_id, product_id))
                inventory = cursor.fetchone()

                # int()/str() at the boundary: psycopg2 hands back Any, so this is the
                # only place the declared types are actually enforced.
                available_qty = int(inventory["available_quantity"]) if inventory else 0

                if inventory and available_qty >= quantity:
                    # Update inventory to reserve the items
                    update_query = """
                        UPDATE warehouses.inventory
                        SET reserved_quantity = reserved_quantity + %s
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(update_query, (quantity, warehouse_id, product_id))

                    reserved_items.append(
                        {
                            "product_id": product_id,
                            "quantity": quantity,
                            "warehouse_id": warehouse_id,
                            "warehouse_name": str(inventory["warehouse_name"]),
                            "warehouse_location": str(inventory["warehouse_location"]),
                        }
                    )
                else:
                    failed_items.append(
                        {
                            "product_id": product_id,
                            "warehouse_id": warehouse_id,
                            "requested": quantity,
                            "available": available_qty,
                            "reason": "insufficient_stock"
                            if inventory
                            else "not_in_warehouse",
                        }
                    )

            # Only commit if all items were successfully reserved. Anything appended
            # to reserved_items before a failure is rolled back with the rest, so it
            # must not appear in the returned payload.
            if failed_items:
                conn.rollback()
                return {"status": "rejected", "failed_items": failed_items}

            conn.commit()
            return {"status": "reserved", "reserved_items": reserved_items}

    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()


In [ ]:
# The availability check above put DE-MUN-01 in warehouses_full_fulfillment for
# this cart (66 / 92 / 77 in stock against 1 / 3 / 9 requested), so the natural
# follow-up is to reserve every line from Munich.
reservation_requests = [
    ReservationRequest(
        warehouse_id="DE-MUN-01",
        product_id="B0BWJMKC31",
        quantity=1,
    ),
    ReservationRequest(
        warehouse_id="DE-MUN-01",
        product_id="B088MG3SSB",
        quantity=3,
    ),
    ReservationRequest(
        warehouse_id="DE-MUN-01",
        product_id="B0B7XH7VSW",
        quantity=9,
    ),
]


In [ ]:
reserve_warehouse_items(reservation_requests)

#### Splitting one cart across two warehouses

Berlin only stocks `B0BWJMKC31` (19 units) out of this cart, so on its own it lands
in `warehouses_partial_fulfillment`. Pairing it with Hamburg, which stocks all three,
covers the order — and exercises the multi-warehouse path through the transaction.

In [ ]:
split_reservation_requests = [
    ReservationRequest(
        warehouse_id="DE-BER-01",
        product_id="B0BWJMKC31",
        quantity=1,
    ),
    ReservationRequest(
        warehouse_id="DE-HAM-01",
        product_id="B088MG3SSB",
        quantity=3,
    ),
    ReservationRequest(
        warehouse_id="DE-HAM-01",
        product_id="B0B7XH7VSW",
        quantity=9,
    ),
]

reserve_warehouse_items(split_reservation_requests)


#### A rejected batch, with both failure reasons

Paris is the closest depot but a poor choice for this cart: it holds 2 units of
`B088MG3SSB` against the 3 requested, and has no `B0BWJMKC31` row at all. That is one
line of each failure kind — and the Marseille line that *would* have succeeded is
rolled back with them, so `status` comes back `"rejected"` with no `reserved_items` key.

In [ ]:
doomed_reservation_requests = [
    ReservationRequest(
        warehouse_id="FR-MAR-01",
        product_id="B0B7XH7VSW",
        quantity=9,
    ),
    ReservationRequest(
        warehouse_id="FR-PAR-01",
        product_id="B088MG3SSB",
        quantity=3,
    ),
    ReservationRequest(
        warehouse_id="FR-PAR-01",
        product_id="B0BWJMKC31",
        quantity=1,
    ),
]

reserve_warehouse_items(doomed_reservation_requests)
